# Podcast Studio — Cloud Voice Engine (free GPU)

This notebook runs the Chatterbox voice engine on Colab's **free GPU** and gives you a
public URL. You paste that URL into your Mac app's `.env` (as `LOCAL_TTS_URL`) and the
voices become fast + cloning works.

## Do this first (important)
**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**
If you skip this you get the slow CPU again.

Then run the three cells below in order (click each ▶). Keep this tab open while you use the app.

### Cell 1 — install everything (~2-3 min)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > T4 GPU, then re-run.'
print('GPU OK:', torch.cuda.get_device_name(0))

!pip install -q chatterbox-tts fastapi uvicorn python-multipart soundfile
# cloudflared gives us a free public URL with no signup
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('\nInstall done.')

### Cell 2 — write the voice service

In [ ]:
service_code = r'''
import io, json, uuid, shutil, subprocess
from pathlib import Path
from fastapi import FastAPI, UploadFile, Form, HTTPException
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
import soundfile as sf

BASE = Path('.'); VOICES = BASE/'voices'; VOICES.mkdir(exist_ok=True)
REG = VOICES/'registry.json'
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

def load_reg():
    return json.loads(REG.read_text()) if REG.exists() else {}
def save_reg(r): REG.write_text(json.dumps(r, indent=2))

_model = None
def get_model():
    global _model
    if _model is None:
        from chatterbox.tts import ChatterboxTTS
        _model = ChatterboxTTS.from_pretrained(device='cuda')
        print('[tts] model loaded on cuda')
    return _model

def to_wav(data, out):
    if shutil.which('ffmpeg'):
        p = subprocess.run(['ffmpeg','-y','-i','pipe:0','-ar','24000','-ac','1',str(out)],
                           input=data, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if p.returncode==0 and Path(out).exists(): return
    Path(out).write_bytes(data)

@app.get('/health')
def health(): return {'ok': True, 'device': 'cuda', 'model_loaded': _model is not None}

@app.get('/voices')
def voices():
    out = [{'id':'default','name':'Chatterbox Default'}]
    for vid, m in load_reg().items(): out.append({'id': vid, 'name': m['name']})
    return {'voices': out}

@app.post('/clone')
async def clone(name: str = Form('My Voice'), file: UploadFile = None):
    if file is None: raise HTTPException(400, 'No audio file.')
    data = await file.read()
    if not data: raise HTTPException(400, 'Empty file.')
    vid = 'v_'+uuid.uuid4().hex[:8]; ref = VOICES/f'{vid}.wav'; to_wav(data, ref)
    reg = load_reg(); reg[vid] = {'name': name[:60], 'file': ref.name}; save_reg(reg)
    return {'id': vid, 'name': reg[vid]['name']}

@app.post('/tts')
async def tts(payload: dict):
    text = (payload.get('text') or '').strip()
    voice_id = payload.get('voice_id') or 'default'
    if not text: raise HTTPException(400, 'No text.')
    model = get_model(); ref = None
    if voice_id != 'default':
        m = load_reg().get(voice_id)
        if not m: raise HTTPException(404, 'Unknown voice')
        ref = str(VOICES/m['file'])
    wav = model.generate(text, audio_prompt_path=ref) if ref else model.generate(text)
    audio = wav.squeeze().detach().cpu().numpy()
    buf = io.BytesIO(); sf.write(buf, audio, model.sr, format='WAV')
    return Response(content=buf.getvalue(), media_type='audio/wav')
'''
with open('app.py','w') as f: f.write(service_code)
print('Wrote app.py')

### Cell 3 — start the engine + get your public URL
When it finishes it prints a line like `YOUR PUBLIC URL: https://something.trycloudflare.com`.
Copy that whole URL — you'll paste it into your Mac app's `.env`.

In [ ]:
import subprocess, time, re, threading, sys

# start the API server
server = subprocess.Popen([sys.executable,'-m','uvicorn','app:app','--host','0.0.0.0','--port','8000'])
time.sleep(6)

# start the public tunnel and capture the URL it prints
tun = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in tun.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

print('\n' + '='*60)
print('  YOUR PUBLIC URL:', url)
print('  Paste this into your Mac .env as:')
print('     LOCAL_TTS_URL=' + (url or '<url>'))
print('  Keep this Colab tab open while using the app.')
print('='*60)

# keep the tunnel alive + warm up the model so the first podcast line is fast
def warm():
    import urllib.request, json
    try:
        req = urllib.request.Request(url+'/tts', data=json.dumps({'text':'Hello, warming up.'}).encode(),
                                     headers={'content-type':'application/json'})
        urllib.request.urlopen(req, timeout=600).read(); print('[warmup] model ready')
    except Exception as e: print('[warmup] skipped:', e)
threading.Thread(target=warm, daemon=True).start()

# block so the cell keeps running (engine stays up)
for line in tun.stdout:
    pass